# LeetCode 2269 — Find the K-Beauty of a Number bằng Python

Link đề: https://leetcode.com/problems/find-the-k-beauty-of-a-number/ · Độ khó: **Easy** · Chủ đề: Math, String, Sliding Window

Skeleton LeetCode:

```python
class Solution:
    def divisorSubstrings(self, num: int, k: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu khái niệm **k-beauty**: đếm các **chuỗi con độ dài `k`** của `num` (viết ở dạng thập phân) mà giá trị của chúng là **ước** của `num`.
2. Biết **cắt chuỗi** `s[i:i+k]`, đổi `str` ↔ `int`, và phép chia lấy dư `%` để kiểm tra "là ước".
3. Viết 4 cách: cắt chuỗi (brute force), tách cửa sổ bằng toán `// 10**i % 10**k`, **cửa sổ trượt** cập nhật dần, và cách dùng generator/`sum` gọn.
4. Xử lý đúng điểm bẫy của đề: chuỗi con có thể có **số 0 đứng đầu** (`"04"` = `4`) và chuỗi con bằng `0` thì **không phải ước** (tránh chia cho `0`).
5. Tự trace, tự test (vét cạn + random) và tránh lỗi hay gặp: sai số cửa sổ, `ZeroDivisionError`, quên đổi sang `int`.

## 2. Đề bài & ví dụ

**K-beauty** của một số nguyên `num` là **số lượng chuỗi con** của `num` (viết ở dạng thập phân) thỏa **đồng thời**:

- có độ dài **đúng `k`**, và
- là **ước** của `num` (tức `num % giá_trị_chuỗi_con == 0`).

Cho `num` và `k`, trả về k-beauty của `num`. Lưu ý:

- Các chuỗi con **có thể có số `0` đứng đầu** (ví dụ `"04"`).
- Chuỗi con có giá trị `0` **không** là ước của bất kỳ số nào.
- Các chuỗi con được tính **theo vị trí**, nên cùng giá trị ở hai vị trí khác nhau thì đếm hai lần.

| Ví dụ | num | k | Output | Giải thích |
|---|---|---|---|---|
| 1 | `240` | `2` | `2` | `"24"` là ước của `240` (`240 = 24·10`); `"40"` là ước (`240 = 40·6`). |
| 2 | `430` | `2` | `1` | `"43"` là ước của `430` (`430 = 43·10`); `"30"` **không** là ước (`430 = 30·14 + 10`). |

Constraints:

- `1 <= num <= 10^9`
- `1 <= k <= num.length` (độ dài thập phân của `num`).

In [1]:
for num, k, expected in [(240, 2, 2), (430, 2, 1)]:
    s = str(num)
    rows = []
    for i in range(len(s) - k + 1):
        sub = s[i:i + k]
        v = int(sub)
        rows.append((sub, v, v != 0 and num % v == 0))
    print(f"num={num}, k={k}, mong đợi={expected}")
    for r in rows:
        print("   chuỗi con", r[0], "giá trị", r[1], "là ước?", r[2])

num=240, k=2, mong đợi=2
   chuỗi con 24 giá trị 24 là ước? True
   chuỗi con 40 giá trị 40 là ước? True
num=430, k=2, mong đợi=1
   chuỗi con 43 giá trị 43 là ước? True
   chuỗi con 30 giá trị 30 là ước? False


Chạy thử ở trên cho thấy: với `430`, `"43"` là ước (`430 % 43 == 0`), còn `"30"` có `430 % 30 = 10 != 0` nên **không** là ước. Vậy k-beauty của `430` với `k = 2` tính theo định nghĩa là `1`.

> Ghi chú: ví dụ thứ hai của đề gốc cũng dùng `num = 430`, `k = 2`; ở đây ta luôn **tin vào phép tính theo định nghĩa** (và có thể đối chiếu lại với trang LeetCode). Cách an toàn nhất là kiểm chứng từng chuỗi con bằng code như ô trên.

Điều quan trọng là cách làm: **liệt kê mọi cửa sổ độ dài `k`**, đổi sang số, bỏ giá trị `0`, rồi kiểm tra `num % giá_trị == 0`.

## 3. Kiến thức nền

### 3.1. Chuỗi thập phân của một số: `str(num)`

`str(240)` là `"240"`. Độ dài `len(str(num))` là số chữ số `L`. Số chuỗi con độ dài `k` (các cửa sổ liên tiếp) là `L - k + 1`.

In [2]:
num = 240
s = str(num)
print(s, len(s), "số cửa sổ khi k=2:", len(s) - 2 + 1)

240 3 số cửa sổ khi k=2: 2


### 3.2. Cắt chuỗi `s[i:i+k]` và đổi sang số

`s[i:i+k]` lấy `k` ký tự bắt đầu từ `i`. `int("04")` là `4` (số `0` đứng đầu bị bỏ). `int("00")` là `0`.

In [3]:
s = "2400"
for i in range(len(s) - 2 + 1):
    sub = s[i:i + 2]
    print(i, repr(sub), int(sub))

0 '24' 24
1 '40' 40
2 '00' 0


### 3.3. Là ước: `num % d == 0` và cẩn thận `d = 0`

`d` là ước của `num` nếu chia hết: `num % d == 0`. Nhưng `num % 0` gây `ZeroDivisionError`, nên **phải kiểm tra `d != 0` trước**. Nhờ **đoản mạch** (short-circuit) của `and`, `d != 0 and num % d == 0` không bao giờ tính `num % 0`.

In [4]:
num = 240
for d in [24, 40, 7, 0]:
    print(d, d != 0 and num % d == 0)
try:
    print(240 % 0)
except ZeroDivisionError as e:
    print("ZeroDivisionError:", e)

24 True
40 True
7 False
0 False
ZeroDivisionError: integer modulo by zero


### 3.4. Lấy cửa sổ bằng toán: `(num // 10**i) % 10**k`

- `num // 10**i` bỏ `i` chữ số cuối.
- `% 10**k` giữ lại `k` chữ số cuối của kết quả đó.

Hai bước kết hợp cho cửa sổ `k` chữ số, kết thúc cách `i` chữ số tính từ bên phải. Cách này **không dùng chuỗi**.

In [5]:
num, k = 2400, 2
for i in range(len(str(num)) - k + 1):
    print("i =", i, "->", (num // 10**i) % 10**k)

i = 0 -> 0
i = 1 -> 40
i = 2 -> 24


### 3.5. Cửa sổ trượt (sliding window) trên dãy chữ số

Thay vì tính lại mỗi cửa sổ từ đầu, ta cập nhật: bỏ chữ số cũ nhất (hàng cao nhất) và thêm chữ số mới ở hàng đơn vị. Với cửa sổ `w` có `k` chữ số: `w = (w % 10**(k-1)) * 10 + chữ_số_mới`. Ví dụ cửa sổ `"24"` → trượt tới `"40"`: `(24 % 10) * 10 + 0 = 40`.

In [6]:
digits = [2, 4, 0]
k = 2
w = digits[0] * 10 + digits[1]
print(w)
w = (w % 10**(k - 1)) * 10 + digits[2]
print(w)

24
40


## 4. Ý tưởng nền tảng — tính bằng tay

Với `num = 240`, `k = 2`: chuỗi `"240"`, các cửa sổ độ dài `2`: `"24"` và `"40"`.

| Cửa sổ | Giá trị | `240 % giá trị` | Là ước? |
|---|---|---|---|
| `"24"` | `24` | `0` | **có** |
| `"40"` | `40` | `0` | **có** |

K-beauty `= 2`.

Với `num = 430`, `k = 2`: cửa sổ `"43"` và `"30"`: `430 % 43 = 0` (có); `430 % 30 = 10` (không) ⇒ k-beauty bằng `1`.

Các bước tay:

1. Viết `num` thành chuỗi `s`, `L = len(s)`.
2. Với `i = 0, 1, ..., L - k`: lấy `s[i:i+k]`, đổi sang `int` được `v`.
3. Nếu `v != 0` và `num % v == 0` thì tăng đếm.

In [7]:
for num, k in [(240, 2), (430, 2), (10, 1), (2400, 2)]:
    s = str(num)
    count = 0
    for i in range(len(s) - k + 1):
        v = int(s[i:i + k])
        if v != 0 and num % v == 0:
            count += 1
    print(f"num={num}, k={k} -> {count}")

num=240, k=2 -> 2
num=430, k=2 -> 1
num=10, k=1 -> 1
num=2400, k=2 -> 2


## 5. Cách 1 — Brute Force: cắt chuỗi con rồi kiểm tra

Đúng như định nghĩa: đổi `num` sang chuỗi, duyệt mọi cửa sổ độ dài `k`, đổi sang số, kiểm tra ước.

In [8]:
class SolutionBrute:
    def divisorSubstrings(self, num, k):
        s = str(num)
        count = 0
        for i in range(len(s) - k + 1):
            v = int(s[i:i + k])
            if v != 0 and num % v == 0:
                count += 1
        return count

sol = SolutionBrute()
print(sol.divisorSubstrings(240, 2))
print(sol.divisorSubstrings(430, 2))
print(sol.divisorSubstrings(2400, 2))

2
1
2


**Độ phức tạp:** gọi `L` là số chữ số (`L <= 10`). Có `L - k + 1` cửa sổ, mỗi cửa sổ cắt/đổi `O(k)` ⇒ Time `O(L · k)`, Space `O(L)` cho chuỗi.

**Nút thắt:** với `L <= 10` thì không có nút thắt thực sự, nhưng tư duy "cập nhật cửa sổ" thay vì "tính lại từ đầu" là mấu chốt cho các bài cửa sổ trượt lớn hơn.

## 6. Ý tưởng thuật toán — bước đột phá

**Insight:** hai cửa sổ liền kề **chỉ khác nhau một chữ số ở mỗi đầu**: cửa sổ mới = bỏ chữ số cao nhất cũ + thêm một chữ số thấp nhất mới. Vì thế ta có công thức cập nhật `O(1)`:

`w_new = (w_old % 10**(k-1)) * 10 + d_new`

Quy tắc "nếu… thì…":

- **Nếu** `w != 0` **và** `num % w == 0` **thì** đếm thêm `1`.
- **Nếu** `w == 0` **thì** bỏ qua (không là ước, tránh chia cho `0`).

**Invariant (cửa sổ trượt):** sau mỗi bước, `w` bằng giá trị số của `k` chữ số liên tiếp hiện tại (có thể có `0` đứng đầu và khi đó `w` nhỏ hơn).

## 7. Các cách tiếp theo

### Cách 2 — Tách cửa sổ bằng toán `//` và `%` (không dùng chuỗi con)

Vẫn cần biết số chữ số `L`; dùng `len(str(num))` một lần. Mỗi cửa sổ: `v = (num // 10**i) % 10**k`.

In [9]:
class SolutionMath:
    def divisorSubstrings(self, num, k):
        L = len(str(num))
        p = 10 ** k
        count = 0
        for i in range(L - k + 1):
            v = (num // 10 ** i) % p
            if v != 0 and num % v == 0:
                count += 1
        return count

sol = SolutionMath()
print(sol.divisorSubstrings(240, 2))
print(sol.divisorSubstrings(430, 2))
print(sol.divisorSubstrings(2400, 2))

2
1
2


**Độ phức tạp:** Time `O(L)` phép toán số học (mỗi phép `//`, `%` với số nhỏ coi là `O(1)`), Space `O(1)`.

### Cách 3 — Cửa sổ trượt với danh sách chữ số

Lấy danh sách chữ số của `num`; tính cửa sổ đầu tiên, sau đó cập nhật theo công thức `w = (w % 10**(k-1)) * 10 + d`.

In [10]:
class SolutionSliding:
    def divisorSubstrings(self, num, k):
        digits = [int(c) for c in str(num)]
        p = 10 ** (k - 1)
        w = 0
        for d in digits[:k]:
            w = w * 10 + d                         # cửa sổ đầu tiên
        count = 1 if (w != 0 and num % w == 0) else 0
        for d in digits[k:]:
            w = (w % p) * 10 + d                   # bỏ chữ số cao nhất, thêm chữ số mới
            if w != 0 and num % w == 0:
                count += 1
        return count

sol = SolutionSliding()
print(sol.divisorSubstrings(240, 2))
print(sol.divisorSubstrings(430, 2))
print(sol.divisorSubstrings(2400, 2))

2
1
2


**Độ phức tạp:** Time `O(L)`, Space `O(L)` cho danh sách chữ số (có thể `O(1)` nếu lấy chữ số trực tiếp từ `num`).

### Cách 4 — Generator + `sum` (một dòng, Pythonic)

`sum(1 for ... if điều_kiện)` đếm số phần tử thỏa điều kiện. Ta dùng biểu thức gán `:=` (Python 3.8+) để tính `v` một lần cho mỗi cửa sổ.

In [11]:
class SolutionOneLiner:
    def divisorSubstrings(self, num, k):
        s = str(num)
        return sum(1 for i in range(len(s) - k + 1)
                   if (v := int(s[i:i + k])) != 0 and num % v == 0)

sol = SolutionOneLiner()
print(sol.divisorSubstrings(240, 2))
print(sol.divisorSubstrings(430, 2))
print(sol.divisorSubstrings(2400, 2))

2
1
2


**Độ phức tạp:** Time `O(L · k)`, Space `O(L)`. Ngắn gọn nhưng khó đọc hơn với người mới; chú ý dấu ngoặc quanh `(v := ...)`.

## 8. TRACE từng bước

Trace cách 3 (cửa sổ trượt) với `num = 2400`, `k = 2`, `digits = [2,4,0,0]`, `p = 10`:

| Bước | Chữ số mới | w trước | w sau | w != 0? | `2400 % w` | đếm |
|---|---|---|---|---|---|---|
| cửa sổ đầu | `2,4` | | `24` | có | `0` | 1 |
| 1 | `0` | `24` | `(24 % 10)*10 + 0 = 40` | có | `0` | 2 |
| 2 | `0` | `40` | `(40 % 10)*10 + 0 = 0` | **không** | bỏ qua | 2 |

K-beauty `= 2`. Hàm dưới in từng bước.

In [12]:
def trace_sliding(num, k):
    digits = [int(c) for c in str(num)]
    p = 10 ** (k - 1)
    w = 0
    for d in digits[:k]:
        w = w * 10 + d
    count = 0
    ok = w != 0 and num % w == 0
    count += ok
    print(f"cửa sổ đầu: w={w} | w != 0: {w != 0} | num % w = {num % w if w else '-'} | là ước: {ok} | count={count}")
    for d in digits[k:]:
        old = w
        w = (w % p) * 10 + d
        ok = w != 0 and num % w == 0
        count += ok
        print(f"thêm {d}: w = ({old} % {p})*10 + {d} = {w} | là ước: {ok} | count={count}")
    return count

print(trace_sliding(2400, 2))

cửa sổ đầu: w=24 | w != 0: True | num % w = 0 | là ước: True | count=1
thêm 0: w = (24 % 10)*10 + 0 = 40 | là ước: True | count=2
thêm 0: w = (40 % 10)*10 + 0 = 0 | là ước: False | count=2
2


In [13]:
print(trace_sliding(430, 2))

cửa sổ đầu: w=43 | w != 0: True | num % w = 0 | là ước: True | count=1
thêm 0: w = (43 % 10)*10 + 0 = 30 | là ước: False | count=1
1


## 9. Code nộp LeetCode

Dùng cách cắt chuỗi cho dễ đọc, dễ đúng (với `L <= 10`, mọi cách đều tức thời).

In [14]:
from typing import List

class Solution:
    def divisorSubstrings(self, num: int, k: int) -> int:
        s = str(num)
        count = 0
        for i in range(len(s) - k + 1):
            v = int(s[i:i + k])
            if v != 0 and num % v == 0:
                count += 1
        return count

print(Solution().divisorSubstrings(240, 2))
print(Solution().divisorSubstrings(430, 2))

2
1


## 10. Vì sao thuật toán đúng?

- **Duyệt đủ cửa sổ:** chuỗi `s` có `L` ký tự, các cửa sổ độ dài `k` bắt đầu tại `i = 0, 1, ..., L - k` (đúng `L - k + 1` cửa sổ, và `k <= L` nên có ít nhất một). Mỗi cửa sổ (theo vị trí) đúng là một chuỗi con cần xét; trùng giá trị ở hai vị trí khác nhau được đếm hai lần, đúng đề.
- **Giá trị cửa sổ:** `int(s[i:i+k])` bỏ số `0` đứng đầu tự nhiên (`"04"` → `4`), đúng với yêu cầu "chuỗi con có thể có số `0` đứng đầu".
- **Điều kiện ước:** `v` là ước của `num` ⇔ `v != 0` và `num % v == 0`. Kiểm tra `v != 0` trước nhờ đoản mạch nên không bao giờ chia cho `0`.
- **Cửa sổ trượt:** `w_new = (w_old % 10**(k-1)) * 10 + d` đúng vì `w_old % 10**(k-1)` loại chữ số cao nhất (hàng `10^(k-1)`), nhân `10` dịch các chữ số còn lại lên một hàng, cộng `d` là chữ số mới ở hàng đơn vị.
- **Cách toán:** `(num // 10**i) % 10**k` lấy `k` chữ số liên tiếp tính từ hàng `10^i`; cửa sổ ở bên trái tương ứng với `i` lớn hơn, nên thứ tự đếm khác nhưng **tập** các cửa sổ như nhau ⇒ cùng số đếm.

## 11. Phân tích Time / Space Complexity

Gọi `L = len(str(num))` (`L <= 10`).

| Cách | Time | Space |
|---|---|---|
| 1. Cắt chuỗi | `O(L · k)` | `O(L)` |
| 2. Toán `//` `%` | `O(L)` | `O(1)` |
| 3. Cửa sổ trượt | `O(L)` | `O(L)` |
| 4. Generator + `sum` | `O(L · k)` | `O(L)` |

Vì `L <= 10` nên tất cả coi như `O(1)`; sự khác biệt chỉ có ý nghĩa khi áp dụng tư duy này cho **dãy dài** (chuỗi/ mảng `10^5` phần tử).

## 12. Bộ test

Test ví dụ đề, edge case, vét cạn so với định nghĩa độc lập (dùng `str` + `int`), rồi random.

In [15]:
solutions = [SolutionBrute(), SolutionMath(), SolutionSliding(), SolutionOneLiner(), Solution()]

for s in solutions:
    assert s.divisorSubstrings(240, 2) == 2
    assert s.divisorSubstrings(430, 2) == 1
print("ví dụ đề: OK")

ví dụ đề: OK


In [16]:
for s in solutions:
    assert s.divisorSubstrings(5, 1) == 1                 # một chữ số: chính nó là ước
    assert s.divisorSubstrings(10, 1) == 1                # "1" là ước, "0" thì không
    assert s.divisorSubstrings(100, 1) == 1               # chỉ "1" là ước; hai số "0" bị bỏ
    assert s.divisorSubstrings(100, 2) == 1               # "10" là ước, "00" = 0 không
    assert s.divisorSubstrings(120, 3) == 1               # k = L: cả số chia hết cho chính nó
    assert s.divisorSubstrings(111, 1) == 3               # cùng giá trị ở 3 vị trí -> đếm 3 lần
    assert s.divisorSubstrings(7, 1) == 1
    assert s.divisorSubstrings(1000000000, 10) == 1       # num lớn nhất, k = L
print("edge case: OK")

edge case: OK


In [17]:
def reference(num, k):
    """Định nghĩa độc lập: mọi chuỗi con độ dài k của dạng thập phân."""
    s = str(num)
    subs = [s[i:i + k] for i in range(len(s) - k + 1)]
    return len([x for x in subs if int(x) > 0 and num % int(x) == 0])

total = 0
for num in range(1, 3000):
    for k in range(1, len(str(num)) + 1):
        expected = reference(num, k)
        for s in solutions:
            assert s.divisorSubstrings(num, k) == expected, (type(s).__name__, num, k)
        total += 1
print("vét cạn num 1..2999, mọi k hợp lệ:", total, "ca x", len(solutions), "cách: OK")

vét cạn num 1..2999, mọi k hợp lệ: 10889 ca x 5 cách: OK


In [18]:
import random
random.seed(2269)
for _ in range(1500):
    num = random.randint(1, 10**9)
    k = random.randint(1, len(str(num)))
    expected = reference(num, k)
    for s in solutions:
        assert s.divisorSubstrings(num, k) == expected, (type(s).__name__, num, k)
print("random đối chiếu 1500 ca x", len(solutions), "cách: OK")

random đối chiếu 1500 ca x 5 cách: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chia cho `0` khi chuỗi con là `"00"` hoặc `"0"`

Số có hai chữ số `0` liên tiếp tạo cửa sổ `0`, và `num % 0` gây `ZeroDivisionError`.

In [19]:
def wrong_zero_division(num, k):
    s = str(num)
    count = 0
    for i in range(len(s) - k + 1):
        v = int(s[i:i + k])
        if num % v == 0:              # SAI: thiếu kiểm tra v != 0
            count += 1
    return count

try:
    print(wrong_zero_division(100, 2))
except ZeroDivisionError as e:
    print("ZeroDivisionError:", e)
print("đúng:", Solution().divisorSubstrings(100, 2))

ZeroDivisionError: integer modulo by zero
đúng: 1


### Lỗi 2 — Sai số cửa sổ: `range(len(s) - k)` thiếu cửa sổ cuối

Số cửa sổ là `L - k + 1`. Quên `+ 1` làm mất cửa sổ cuối cùng.

In [20]:
def wrong_range(num, k):
    s = str(num)
    count = 0
    for i in range(len(s) - k):       # SAI: thiếu + 1
        v = int(s[i:i + k])
        if v != 0 and num % v == 0:
            count += 1
    return count

print("sai :", wrong_range(240, 2), "(bỏ sót cửa sổ cuối '40')")
print("đúng:", Solution().divisorSubstrings(240, 2))
print("k = L:", wrong_range(120, 3), "vs đúng", Solution().divisorSubstrings(120, 3))

sai : 1 (bỏ sót cửa sổ cuối '40')
đúng: 2
k = L: 0 vs đúng 1


### Lỗi 3 — Quên đổi chuỗi con sang `int`

`num % "24"` gây `TypeError` (không chia số cho chuỗi).

In [21]:
def wrong_no_int(num, k):
    s = str(num)
    return sum(1 for i in range(len(s) - k + 1) if num % s[i:i + k] == 0)

try:
    wrong_no_int(240, 2)
except TypeError as e:
    print("TypeError:", e)
print("đúng:", Solution().divisorSubstrings(240, 2))

TypeError: unsupported operand type(s) for %: 'int' and 'str'
đúng: 2


### Lỗi 4 — Đếm **giá trị** khác nhau (dùng `set`) thay vì theo **vị trí**

Đề đếm theo vị trí. `111` với `k = 1` có ba cửa sổ giống nhau ⇒ đáp án `3`, không phải `1`.

In [22]:
def wrong_set(num, k):
    s = str(num)
    vals = {int(s[i:i + k]) for i in range(len(s) - k + 1)}
    return sum(1 for v in vals if v != 0 and num % v == 0)

print("sai :", wrong_set(111, 1))       # 1
print("đúng:", Solution().divisorSubstrings(111, 1))   # 3

sai : 1
đúng: 3


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chuỗi con độ dài `k` của số" ⇒ nghĩ ngay **cửa sổ trượt**: có `L - k + 1` cửa sổ.
- Mọi lúc dùng phép chia/modulo với giá trị lấy từ dữ liệu, hãy tự hỏi: **có thể bằng `0` không?**
- `int("007")` bằng `7`: số `0` đứng đầu tự nhiên bị bỏ khi đổi sang số.
- Để lấy `k` chữ số mà không dùng chuỗi: `(num // 10**i) % 10**k`.
- Cập nhật cửa sổ: `w = (w % 10**(k-1)) * 10 + chữ_số_mới`.
- Họ hàng: Maximum Average Subarray I (643), Number of Substrings, Repeated Substring Pattern, Minimum Window Substring (76), Maximum Number of Vowels in a Substring of Given Length (1456).

## 15. Biến thể của bài

1. **Chuỗi con độ dài bất kỳ:** thử mọi `k` từ `1` đến `L`.
2. **Là bội thay vì là ước:** `v % ... `/`v` chia hết cho một số cho trước.
3. **Liệt kê** các chuỗi con thỏa (không chỉ đếm).
4. **Số trong hệ cơ số khác:** chuỗi con ở hệ nhị phân.
5. **Dãy rất dài (chuỗi chữ số `10^5`):** dùng cửa sổ trượt + giữ số dư bằng modular arithmetic.

In [23]:
# Biến thể 1 & 3: liệt kê các chuỗi con (mọi độ dài) là ước của num
def all_divisor_substrings(num):
    s = str(num)
    out = []
    for k in range(1, len(s) + 1):
        for i in range(len(s) - k + 1):
            v = int(s[i:i + k])
            if v != 0 and num % v == 0:
                out.append(s[i:i + k])
    return out

print(all_divisor_substrings(240))

['2', '4', '24', '40', '240']


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Cắt chuỗi | `int(s[i:i+k])` từng cửa sổ | `O(L·k)` | `O(L)` | **Nộp bài**, dễ đọc, dễ đúng |
| 2. Toán `//` `%` | `(num // 10**i) % 10**k` | `O(L)` | `O(1)` | Không muốn dùng chuỗi con |
| 3. Cửa sổ trượt | Cập nhật `w` theo từng chữ số | `O(L)` | `O(L)` | Học kỹ thuật cho dãy dài |
| 4. Generator + `sum` | Gọn một biểu thức | `O(L·k)` | `O(L)` | Viết nhanh, phỏng vấn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `Solution().divisorSubstrings(120, 2)` là bao nhiêu?

**Bài 2 (tính tay).** Với `num = 1224`, `k = 2`, liệt kê các cửa sổ và kiểm tra từng cái có là ước không.

**Bài 3 (điền TODO).** Hoàn thành `windows(s, k)` trả về danh sách các cửa sổ (chuỗi con độ dài `k`).

**Bài 4 (tự viết lại).** Viết `k_beauty_math(num, k)` chỉ dùng `//` và `%` (không cắt chuỗi con).

**Bài 5 (mini challenge).** Viết `max_k_beauty(num)`: giá trị `k` (từ `1` đến `L`) cho k-beauty **lớn nhất** (nếu hòa, chọn `k` nhỏ nhất).

In [24]:
# Bài 1: đoán rồi chạy
print(Solution().divisorSubstrings(120, 2))

2


In [25]:
# Bài 3, 4, 5: TODO - KHÔNG được gọi khi run all
def windows(s, k):
    # TODO: trả về danh sách s[i:i+k] cho mọi i hợp lệ
    pass

def k_beauty_math(num, k):
    # TODO: chỉ dùng // và %, không cắt chuỗi con
    pass

def max_k_beauty(num):
    # TODO: trả về k (1..L) cho k-beauty lớn nhất; hòa chọn k nhỏ nhất
    pass

## 18. Gợi ý 3 mức

**Bài 3**
- Mức 1: Có `len(s) - k + 1` cửa sổ.
- Mức 2: Cửa sổ thứ `i` là `s[i:i + k]`.
- Mức 3: `return [s[i:i + k] for i in range(len(s) - k + 1)]`.

**Bài 4**
- Mức 1: Số chữ số `L = len(str(num))` (được phép dùng một lần).
- Mức 2: Cửa sổ ở hàng `i`: `(num // 10**i) % 10**k`.
- Mức 3: Đếm khi `v != 0 and num % v == 0`.

**Bài 5**
- Mức 1: Thử mọi `k` từ `1` đến `L`.
- Mức 2: Dùng `Solution().divisorSubstrings(num, k)` cho mỗi `k`.
- Mức 3: Duyệt `k` tăng dần, chỉ cập nhật khi giá trị **lớn hơn hẳn** để hòa giữ `k` nhỏ.

## 19. Đáp án tham khảo cho bài tập

Mỗi đáp án được kiểm tra bằng `assert`.

In [26]:
# Bài 1: "12" là ước của 120 (120 = 12*10); "20" là ước (120 = 20*6) -> 2
assert Solution().divisorSubstrings(120, 2) == 2

# Bài 2: 1224, k=2 -> "12", "22", "24": 1224%12=0, 1224%22=14, 1224%24=0 -> 2
assert [1224 % 12, 1224 % 22, 1224 % 24] == [0, 14, 0]
assert Solution().divisorSubstrings(1224, 2) == 2

# Bài 3
def windows_ans(s, k):
    return [s[i:i + k] for i in range(len(s) - k + 1)]
assert windows_ans("1224", 2) == ["12", "22", "24"]
assert windows_ans("5", 1) == ["5"]

# Bài 4
def k_beauty_math_ans(num, k):
    L = len(str(num))
    count = 0
    for i in range(L - k + 1):
        v = (num // 10 ** i) % 10 ** k
        if v != 0 and num % v == 0:
            count += 1
    return count
assert k_beauty_math_ans(240, 2) == 2
assert k_beauty_math_ans(2400, 2) == 2
assert k_beauty_math_ans(111, 1) == 3

# Bài 5
def max_k_beauty_ans(num):
    best_k, best = 1, -1
    for k in range(1, len(str(num)) + 1):
        v = Solution().divisorSubstrings(num, k)
        if v > best:
            best_k, best = k, v
    return best_k
assert max_k_beauty_ans(111) == 1          # k=1 cho 3, k=2 cho 2 ("11" không chia hết 111), k=3 cho 1
assert max_k_beauty_ans(5) == 1
print("Tất cả đáp án bài tập: OK")

Tất cả đáp án bài tập: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

### Template

```python
class Solution:
    def divisorSubstrings(self, num: int, k: int) -> int:
        s = str(num)
        count = 0
        for i in range(len(s) - k + 1):        # L - k + 1 cửa sổ
            v = int(s[i:i + k])
            if v != 0 and num % v == 0:        # kiểm tra v != 0 TRƯỚC khi chia
                count += 1
        return count
```

### Checklist

- [ ] Số cửa sổ là `len(s) - k + 1` (không bỏ sót cửa sổ cuối).
- [ ] Đổi chuỗi con sang `int` trước khi chia.
- [ ] Kiểm tra `v != 0` để tránh `ZeroDivisionError`; chuỗi con `0` không là ước.
- [ ] Đếm theo **vị trí** (giá trị lặp ở nhiều vị trí đếm nhiều lần).
- [ ] Đã test `"00"`, `k = L`, `k = 1`, số có chữ số `0` đứng đầu cửa sổ (như `"04"`).
- [ ] Biết cách cửa sổ trượt cho dãy dài.